In [1]:
import pandas as pd
from pathlib import Path

In [2]:
base_dir = Path.cwd().parent
data_dir = base_dir / "data/raw"

dfs = {}

for file in data_dir.glob("*.csv"):
    name = file.stem.replace("olist_", "").replace("_dataset", "")
    dfs[name] = pd.read_csv(file)

In [3]:
orders = dfs["orders"]
customers = dfs["customers"]
geolocation = dfs["geolocation"]
sellers = dfs["sellers"]
products = dfs["products"]
category_name_translation = dfs["product_category_name_translation"]
order_items = dfs["order_items"]
order_payments = dfs["order_payments"]
order_reviews = dfs["order_reviews"]

# Orders

Non ci sono righe duplicate.

Le seguenti colonne presentano valori null:

* *order_approved_at*
* *order_delivered_carrier_date*
* *order_delivered_customer_date*

Le seguenti colonne vanno convertite in formato datetime:

* *order_purchase_timestamp*
* *order_approved_at*
* *order_delivered_carrier_date*
* *order_delivered_customer_date*
* *order_estimated_delivery_date*

Gli ordini che hanno valori mancanti in *order_approved_at* sono ordini *canceled*, anche se alcuni sono in stato *created* o *delivered*.

In [4]:
orders.info(), orders.shape, orders.head(), orders.nunique()

<class 'pandas.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 8 columns):
 #   Column                         Non-Null Count  Dtype
---  ------                         --------------  -----
 0   order_id                       99441 non-null  str  
 1   customer_id                    99441 non-null  str  
 2   order_status                   99441 non-null  str  
 3   order_purchase_timestamp       99441 non-null  str  
 4   order_approved_at              99281 non-null  str  
 5   order_delivered_carrier_date   97658 non-null  str  
 6   order_delivered_customer_date  96476 non-null  str  
 7   order_estimated_delivery_date  99441 non-null  str  
dtypes: str(8)
memory usage: 6.1 MB


(None,
 (99441, 8),
                            order_id                       customer_id  \
 0  e481f51cbdc54678b7cc49136f2d6af7  9ef432eb6251297304e76186b10a928d   
 1  53cdb2fc8bc7dce0b6741e2150273451  b0830fb4747a6c6d20dea0b8c802d7ef   
 2  47770eb9100c2d0c44946d9cf07ec65d  41ce2a54c0b03bf3443c3d931a367089   
 3  949d5b44dbf5de918fe9c16f97b45f8a  f88197465ea7920adcdbec7375364d82   
 4  ad21c59c0840e6cb83a9ceb5573f8159  8ab97904e6daea8866dbdbc4fb7aad2c   
 
   order_status order_purchase_timestamp    order_approved_at  \
 0    delivered      2017-10-02 10:56:33  2017-10-02 11:07:15   
 1    delivered      2018-07-24 20:41:37  2018-07-26 03:24:27   
 2    delivered      2018-08-08 08:38:49  2018-08-08 08:55:23   
 3    delivered      2017-11-18 19:28:06  2017-11-18 19:45:59   
 4    delivered      2018-02-13 21:18:39  2018-02-13 22:20:29   
 
   order_delivered_carrier_date order_delivered_customer_date  \
 0          2017-10-04 19:55:00           2017-10-10 21:25:13   
 1          

In [5]:
orders.query(
    "order_approved_at.isnull()"
)

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
1130,00b1cb0320190ca0daa2c88b35206009,3532ba38a3fd242259a514ac2b6ae6b6,canceled,2018-08-28 15:26:39,NaN,NaN,NaN,2018-09-12 00:00:00
1801,ed3efbd3a87bea76c2812c66a0b32219,191984a8ba4cbb2145acb4fe35b69664,canceled,2018-09-20 13:54:16,NaN,NaN,NaN,2018-10-17 00:00:00
1868,df8282afe61008dc26c6c31011474d02,aa797b187b5466bc6925aaaa4bb3bed1,canceled,2017-03-04 12:14:30,NaN,NaN,NaN,2017-04-10 00:00:00
2029,8d4c637f1accf7a88a4555f02741e606,b1dd715db389a2077f43174e7a675d07,canceled,2018-08-29 16:27:49,NaN,NaN,NaN,2018-09-13 00:00:00
2161,7a9d4c7f9b068337875b95465330f2fc,7f71ae48074c0cfec9195f88fcbfac55,canceled,2017-05-01 16:12:39,NaN,NaN,NaN,2017-05-30 00:00:00
...,...,...,...,...,...,...,...,...
97696,5a00b4d35edffc56b825c3646a99ba9d,6a3bdf004ca96338fb5fad1b8d93c2e6,canceled,2017-07-02 15:38:46,NaN,NaN,NaN,2017-07-25 00:00:00
98415,227c804e2a44760671a6a5697ea549e4,62e7477e75e542243ee62a0ba73f410f,canceled,2017-09-28 15:02:56,NaN,NaN,NaN,2017-10-16 00:00:00
98909,e49e7ce1471b4693482d40c2bd3ad196,e4e7ab3f449aeb401f0216f86c2104db,canceled,2018-08-07 11:16:28,NaN,NaN,NaN,2018-08-10 00:00:00
99283,3a3cddda5a7c27851bd96c3313412840,0b0d6095c5555fe083844281f6b093bb,canceled,2018-08-31 16:13:44,NaN,NaN,NaN,2018-10-01 00:00:00


In [6]:
orders.query(
    "order_approved_at.isnull() and order_status != 'canceled'"
).sort_values("order_status")

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
7434,b5359909123fa03c50bdb0cfed07f098,438449d4af8980d107bf04571413a8e7,created,2017-12-05 01:07:52,NaN,NaN,NaN,2018-01-11 00:00:00
9238,dba5062fbda3af4fb6c33b1e040ca38f,964a6df3d9bdf60fe3e7b8bb69ed893a,created,2018-02-09 17:21:04,NaN,NaN,NaN,2018-03-07 00:00:00
21441,7a4df5d8cff4090e541401a20a22bb80,725e9c75605414b21fd8c8d5a1c2f1d6,created,2017-11-25 11:10:33,NaN,NaN,NaN,2017-12-12 00:00:00
58958,90ab3e7d52544ec7bc3363c82689965f,7d61b9f4f216052ba664f22e9c504ef1,created,2017-11-06 13:12:34,NaN,NaN,NaN,2017-12-01 00:00:00
55086,35de4050331c6c644cddc86f4f2d0d64,4ee64f4bfc542546f422da0aeb462853,created,2017-12-05 01:07:58,NaN,NaN,NaN,2018-01-08 00:00:00
5323,e04abd8149ef81b95221e88f6ed9ab6a,2127dc6603ac33544953ef05ec155771,delivered,2017-02-18 14:40:00,NaN,2017-02-23 12:04:47,2017-03-01 13:25:33,2017-03-17 00:00:00
67697,88083e8f64d95b932164187484d90212,f67cd1a215aae2a1074638bbd35a223a,delivered,2017-02-18 22:49:19,NaN,2017-02-22 11:31:06,2017-03-02 12:06:06,2017-03-21 00:00:00
63052,51eb2eebd5d76a24625b31c33dd41449,07a2a7e0f63fd8cb757ed77d4245623c,delivered,2017-02-18 15:52:27,NaN,2017-02-23 03:09:14,2017-03-07 13:57:47,2017-03-29 00:00:00
61743,2eecb0d85f281280f79fa00f9cec1a95,a3d3c38e58b9d2dfb9207cab690b6310,delivered,2017-02-17 17:21:55,NaN,2017-02-22 11:42:51,2017-03-03 12:16:03,2017-03-20 00:00:00
48401,7002a78c79c519ac54022d4f8a65e6e8,d5de688c321096d15508faae67a27051,delivered,2017-01-19 22:26:59,NaN,2017-01-27 11:08:05,2017-02-06 14:22:19,2017-03-16 00:00:00


# Customers

Non ci sono righe duplicate o valori null.

In [7]:
customers.info(), customers.shape, customers.head(), customers.nunique()

<class 'pandas.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 5 columns):
 #   Column                    Non-Null Count  Dtype
---  ------                    --------------  -----
 0   customer_id               99441 non-null  str  
 1   customer_unique_id        99441 non-null  str  
 2   customer_zip_code_prefix  99441 non-null  int64
 3   customer_city             99441 non-null  str  
 4   customer_state            99441 non-null  str  
dtypes: int64(1), str(4)
memory usage: 3.8 MB


(None,
 (99441, 5),
                         customer_id                customer_unique_id  \
 0  06b8999e2fba1a1fbc88172c00ba8bc7  861eff4711a542e4b93843c6dd7febb0   
 1  18955e83d337fd6b2def6b18a428ac77  290c77bc529b7ac935b93aa66c333dc3   
 2  4e7b3e00288586ebd08712fdd0374a03  060e732b5b29e8181a18229c7b0b2b5e   
 3  b2b6027bc5c5109e529d4dc6358b12c3  259dac757896d24d7702b9acbbff3f3c   
 4  4f2d8ab171c80ec8364f7c12e35b23ad  345ecd01c38d18a9036ed96c73b8d066   
 
    customer_zip_code_prefix          customer_city customer_state  
 0                     14409                 franca             SP  
 1                      9790  sao bernardo do campo             SP  
 2                      1151              sao paulo             SP  
 3                      8775        mogi das cruzes             SP  
 4                     13056               campinas             SP  ,
 customer_id                 99441
 customer_unique_id          96096
 customer_zip_code_prefix    14994
 customer_city

# Geolocation

Ci sono 261.831 righe duplicate.

Non ci sono valori null.

Le seguenti colonne presentano valori negativi che non vanno corretti:

* *geolocation_lat*
* *geolocation_lng*

In [8]:
geolocation.info(), geolocation.shape, geolocation.head(), geolocation.nunique()

<class 'pandas.DataFrame'>
RangeIndex: 1000163 entries, 0 to 1000162
Data columns (total 5 columns):
 #   Column                       Non-Null Count    Dtype  
---  ------                       --------------    -----  
 0   geolocation_zip_code_prefix  1000163 non-null  int64  
 1   geolocation_lat              1000163 non-null  float64
 2   geolocation_lng              1000163 non-null  float64
 3   geolocation_city             1000163 non-null  str    
 4   geolocation_state            1000163 non-null  str    
dtypes: float64(2), int64(1), str(2)
memory usage: 38.2 MB


(None,
 (1000163, 5),
    geolocation_zip_code_prefix  geolocation_lat  geolocation_lng  \
 0                         1037       -23.545621       -46.639292   
 1                         1046       -23.546081       -46.644820   
 2                         1046       -23.546129       -46.642951   
 3                         1041       -23.544392       -46.639499   
 4                         1035       -23.541578       -46.641607   
 
   geolocation_city geolocation_state  
 0        sao paulo                SP  
 1        sao paulo                SP  
 2        sao paulo                SP  
 3        sao paulo                SP  
 4        sao paulo                SP  ,
 geolocation_zip_code_prefix     19015
 geolocation_lat                717360
 geolocation_lng                717613
 geolocation_city                 8011
 geolocation_state                  27
 dtype: int64)

In [9]:
geolocation.duplicated().sum()

np.int64(261831)

# Sellers

Non ci sono righe duplicate o valori null.

In [10]:
sellers.info(), sellers.shape, sellers.head(), sellers.nunique()

<class 'pandas.DataFrame'>
RangeIndex: 3095 entries, 0 to 3094
Data columns (total 4 columns):
 #   Column                  Non-Null Count  Dtype
---  ------                  --------------  -----
 0   seller_id               3095 non-null   str  
 1   seller_zip_code_prefix  3095 non-null   int64
 2   seller_city             3095 non-null   str  
 3   seller_state            3095 non-null   str  
dtypes: int64(1), str(3)
memory usage: 96.8 KB


(None,
 (3095, 4),
                           seller_id  seller_zip_code_prefix  \
 0  3442f8959a84dea7ee197c632cb2df15                   13023   
 1  d1b65fc7debc3361ea86b5f14c68d2e2                   13844   
 2  ce3ad9de960102d0677a81f5d0bb7b2d                   20031   
 3  c0f3eea2e14555b6faeea3dd58c1b1c3                    4195   
 4  51a04a8a6bdcb23deccc82b0b80742cf                   12914   
 
          seller_city seller_state  
 0           campinas           SP  
 1         mogi guacu           SP  
 2     rio de janeiro           RJ  
 3          sao paulo           SP  
 4  braganca paulista           SP  ,
 seller_id                 3095
 seller_zip_code_prefix    2246
 seller_city                611
 seller_state                23
 dtype: int64)

# Products

Non ci sono righe duplicate.

Ogni colonna, ad eccezione di *product_id*, presenta valori null.

Le seguenti colonne presentano un errore ortografico ("lenght" invece di "length") e va corretto:

* *product_name_lenght*
* *product_description_lenght*

In [21]:
products.info(), products.shape, products.head(), products.nunique()

<class 'pandas.DataFrame'>
RangeIndex: 32951 entries, 0 to 32950
Data columns (total 9 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   product_id                  32951 non-null  str    
 1   product_category_name       32341 non-null  str    
 2   product_name_lenght         32341 non-null  float64
 3   product_description_lenght  32341 non-null  float64
 4   product_photos_qty          32341 non-null  float64
 5   product_weight_g            32949 non-null  float64
 6   product_length_cm           32949 non-null  float64
 7   product_height_cm           32949 non-null  float64
 8   product_width_cm            32949 non-null  float64
dtypes: float64(7), str(2)
memory usage: 2.3 MB


(None,
 (32951, 9),
                          product_id  product_category_name  \
 0  1e9e8ef04dbcff4541ed26657ea517e5             perfumaria   
 1  3aa071139cb16b67ca9e5dea641aaa2f                  artes   
 2  96bd76ec8810374ed1b65e291975717f          esporte_lazer   
 3  cef67bcfe19066a932b7673e239eb23d                  bebes   
 4  9dc1a7de274444849c219cff195d0b71  utilidades_domesticas   
 
    product_name_lenght  product_description_lenght  product_photos_qty  \
 0                 40.0                       287.0                 1.0   
 1                 44.0                       276.0                 1.0   
 2                 46.0                       250.0                 1.0   
 3                 27.0                       261.0                 1.0   
 4                 37.0                       402.0                 4.0   
 
    product_weight_g  product_length_cm  product_height_cm  product_width_cm  
 0             225.0               16.0               10.0           

# Category Name Translation

Non ci sono righe duplicate e valori null.

In [33]:
category_name_translation.info(), category_name_translation.shape,\
category_name_translation.head(), category_name_translation.nunique()

<class 'pandas.DataFrame'>
RangeIndex: 71 entries, 0 to 70
Data columns (total 2 columns):
 #   Column                         Non-Null Count  Dtype
---  ------                         --------------  -----
 0   product_category_name          71 non-null     str  
 1   product_category_name_english  71 non-null     str  
dtypes: str(2)
memory usage: 1.2 KB


(None,
 (71, 2),
     product_category_name product_category_name_english
 0            beleza_saude                 health_beauty
 1  informatica_acessorios         computers_accessories
 2              automotivo                          auto
 3         cama_mesa_banho                bed_bath_table
 4        moveis_decoracao               furniture_decor,
 product_category_name            71
 product_category_name_english    71
 dtype: int64)

# Order Items

Non ci sono righe duplicate e valori null.

La colonna *shipping_limit_date* va convertita in formato datetime.

In [35]:
order_items.info(), order_items.shape, order_items.head(), order_items.nunique()

<class 'pandas.DataFrame'>
RangeIndex: 112650 entries, 0 to 112649
Data columns (total 7 columns):
 #   Column               Non-Null Count   Dtype  
---  ------               --------------   -----  
 0   order_id             112650 non-null  str    
 1   order_item_id        112650 non-null  int64  
 2   product_id           112650 non-null  str    
 3   seller_id            112650 non-null  str    
 4   shipping_limit_date  112650 non-null  str    
 5   price                112650 non-null  float64
 6   freight_value        112650 non-null  float64
dtypes: float64(2), int64(1), str(4)
memory usage: 6.0 MB


(None,
 (112650, 7),
                            order_id  order_item_id  \
 0  00010242fe8c5a6d1ba2dd792cb16214              1   
 1  00018f77f2f0320c557190d7a144bdd3              1   
 2  000229ec398224ef6ca0657da4fc703e              1   
 3  00024acbcdf0a6daa1e931b038114c75              1   
 4  00042b26cf59d7ce69dfabb4e55b4fd9              1   
 
                          product_id                         seller_id  \
 0  4244733e06e7ecb4970a6e2683c13e61  48436dade18ac8b2bce089ec2a041202   
 1  e5f2d52b802189ee658865ca93d83a8f  dd7ddc04e1b6c2c614352b383efe2d36   
 2  c777355d18b72b67abbeef9df44fd0fd  5b51032eddd242adc84c38acab88f23d   
 3  7634da152a4610f1595efa32f14722fc  9d7a1d34a5052409006425275ba1c2b4   
 4  ac6c3623068f30de03045865e4e10089  df560393f3a51e74553ab94004ba5c87   
 
    shipping_limit_date   price  freight_value  
 0  2017-09-19 09:45:35   58.90          13.29  
 1  2017-05-03 11:05:13  239.90          19.93  
 2  2018-01-18 14:48:30  199.00          17.87  
 3  2

In [37]:
order_items.duplicated().sum()

np.int64(0)

In [42]:
pd.to_datetime(
    order_items["shipping_limit_date"],
    errors="coerce"
).isna().sum()

np.int64(0)

# Order Payments

Non ci sono righe duplicate e valori null.

La colonna *payment_type* presenta dei valori *not_defined* che vanno gestiti.

La colonna *payment_installments* presenta valori pari a 0 che vanno gestiti in quanto il numero di rate minimo accettato è 1, ossia il pagamento in un'unica soluzione.

In [45]:
order_payments.info(), order_payments.shape,\
order_payments.head(), order_payments.nunique()

<class 'pandas.DataFrame'>
RangeIndex: 103886 entries, 0 to 103885
Data columns (total 5 columns):
 #   Column                Non-Null Count   Dtype  
---  ------                --------------   -----  
 0   order_id              103886 non-null  str    
 1   payment_sequential    103886 non-null  int64  
 2   payment_type          103886 non-null  str    
 3   payment_installments  103886 non-null  int64  
 4   payment_value         103886 non-null  float64
dtypes: float64(1), int64(2), str(2)
memory usage: 4.0 MB


(None,
 (103886, 5),
                            order_id  payment_sequential payment_type  \
 0  b81ef226f3fe1789b1e8b2acac839d17                   1  credit_card   
 1  a9810da82917af2d9aefd1278f1dcfa0                   1  credit_card   
 2  25e8ea4e93396b6fa0d3dd708e76c1bd                   1  credit_card   
 3  ba78997921bbcdc1373bb41e913ab953                   1  credit_card   
 4  42fdf880ba16b47b59251dd489d4441a                   1  credit_card   
 
    payment_installments  payment_value  
 0                     8          99.33  
 1                     1          24.39  
 2                     1          65.71  
 3                     8         107.78  
 4                     2         128.45  ,
 order_id                99440
 payment_sequential         29
 payment_type                5
 payment_installments       24
 payment_value           29077
 dtype: int64)

In [56]:
order_payments.duplicated().sum()

np.int64(0)

In [63]:
order_payments["payment_type"].value_counts()

payment_type
credit_card    76795
boleto         19784
voucher         5775
debit_card      1529
not_defined        3
Name: count, dtype: int64

In [54]:
order_payments.query("payment_installments < 1")

,order_id,payment_sequential,payment_type,payment_installments,payment_value
46982,744bade1fcf9ff3f31d860ace076d422,2,credit_card,0,58.69
79014,1a57108394169c0b47d8f876acc9ba2d,2,credit_card,0,129.94


In [66]:
order_payments.query("payment_type == 'credit_card'")

,order_id,payment_sequential,payment_type,payment_installments,payment_value
0,b81ef226f3fe1789b1e8b2acac839d17,1,credit_card,8,99.33
1,a9810da82917af2d9aefd1278f1dcfa0,1,credit_card,1,24.39
2,25e8ea4e93396b6fa0d3dd708e76c1bd,1,credit_card,1,65.71
3,ba78997921bbcdc1373bb41e913ab953,1,credit_card,8,107.78
4,42fdf880ba16b47b59251dd489d4441a,1,credit_card,2,128.45
...,...,...,...,...,...
103879,c45067032fd84f4cf408730ff5205568,1,credit_card,2,198.94
103880,7159096c5aa9be77f7f0c26c01ee9793,1,credit_card,4,280.65
103882,7b905861d7c825891d6347454ea7863f,1,credit_card,2,96.80
103883,32609bbb3dd69b3c066a6860554a77bf,1,credit_card,1,47.77


# Order Reviews

Non ci sono righe duplicate.

Le seguenti colonne contengono valori null:

* *review_comment_title*
* *review_comment_message*

Le seguenti colonne vanno convertite in datetime:

* *review_creation_date*
* *review_answer_timestamp*

In [68]:
order_reviews.info(), order_reviews.shape, order_reviews.head(), order_reviews.nunique()

<class 'pandas.DataFrame'>
RangeIndex: 99224 entries, 0 to 99223
Data columns (total 7 columns):
 #   Column                   Non-Null Count  Dtype
---  ------                   --------------  -----
 0   review_id                99224 non-null  str  
 1   order_id                 99224 non-null  str  
 2   review_score             99224 non-null  int64
 3   review_comment_title     11568 non-null  str  
 4   review_comment_message   40977 non-null  str  
 5   review_creation_date     99224 non-null  str  
 6   review_answer_timestamp  99224 non-null  str  
dtypes: int64(1), str(6)
memory usage: 5.3 MB


(None,
 (99224, 7),
                           review_id                          order_id  \
 0  7bc2406110b926393aa56f80a40eba40  73fc7af87114b39712e6da79b0a377eb   
 1  80e641a11e56f04c1ad469d5645fdfde  a548910a1c6147796b98fdf73dbeba33   
 2  228ce5500dc1d8e020d8d1322874b6f0  f9e4b658b201a9f2ecdecbb34bed034b   
 3  e64fb393e7b32834bb789ff8bb30750e  658677c97b385a9be170737859d3511b   
 4  f7c4243c7fe1938f181bec41a392bdeb  8e6bfb81e283fa7e4f11123a3fb894f1   
 
    review_score review_comment_title  \
 0             4                  NaN   
 1             5                  NaN   
 2             5                  NaN   
 3             5                  NaN   
 4             5                  NaN   
 
                               review_comment_message review_creation_date  \
 0                                                NaN  2018-01-18 00:00:00   
 1                                                NaN  2018-03-10 00:00:00   
 2                                                Na

In [70]:
order_reviews.duplicated().sum()

np.int64(0)

In [76]:
pd.to_datetime(
    order_reviews["review_creation_date"],
    errors="coerce"
).isna().sum()

np.int64(0)

In [77]:
pd.to_datetime(
    order_reviews["review_answer_timestamp"],
    errors="coerce"
).isna().sum()

np.int64(0)